# Portfolio sintetico: baseline classica con SCIP

Analisi degli artifact persistiti, senza eseguire la pipeline nel notebook. Il problema è lo stesso del notebook multi-round: 30 asset per impostazione predefinita. Nessun round di riduzione o solver MIS viene eseguito.

```bash
poetry install --with notebooks,experiments
poetry run python -m qreduce.experiments.portfolio_classical --run-id 001_classical --asset-count 30
```

Usare `--asset-count 60` per un'istanza più grande e un nuovo `--run-id` per ogni pubblicazione. Per il confronto usare lo stesso numero di asset dell'esperimento multi-round. SCIP richiede PySCIPOpt; un backend non disponibile può produrre una failure normalizzata.

In [5]:
from pathlib import Path

from qreduce.experiments import load_experiment_run
from qreduce.notebooks.presentation import display_vertical

repository_root = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from inside the QReduce repository.")

run_directory = repository_root / "outputs/001_classical"
artifacts = load_experiment_run(run_directory)
if artifacts.manifest["experiment_id"] != "synthetic-portfolio-classical" or not artifacts.is_classical_only:
    raise ValueError("The selected artifact is not a classical-only portfolio run.")
result = artifacts.result

## 1. Configurazione, provenance e stato

In [6]:
display_vertical(dict(artifacts.manifest), title="Manifest")
display_vertical(dict(artifacts.config["classical_solve"]), title="Solver classico")
display_vertical(dict(result["run_result"]), title="Risultato normalizzato")
display_vertical(dict(result["metrics"]), title="Metriche complessive")

Manifest


,campo,valore
0,artifact_schema_version,1.1
1,config_hash,7fa3c61afcb0393badf8654b0280ab00a0bf7ae8714e56...
2,created_at,2026-11-13T17:09:57.986143Z
3,experiment_id,synthetic-portfolio-classical
4,files,"{\n ""config_used.json"": ""7fa3c61afcb0393badf8..."
5,fixing_manifest,"{\n ""entries"": [],\n ""excluded_item_ids"": []..."
6,has_reduction,False
7,is_classical_only,True
8,model_export,None
9,policy_provenance,"{\n ""declared_instance_class"": null,\n ""decl..."


Solver classico


,campo,valore
0,classical_solver_id,scip
1,integrality_tolerance,0.000001
2,mip_gap_tolerance,None
3,model_export_enabled,False
4,model_export_path,None
5,objective_recomputation_absolute_tolerance,0.000001
6,objective_recomputation_relative_tolerance,0.000001
7,postcheck_mode,strict
8,primal_feasibility_tolerance,0.000001
9,random_seed,7


Risultato normalizzato


,campo,valore
0,adoption_status,adopted
1,classical_solver_status,optimal
2,config_hash,7fa3c61afcb0393badf8654b0280ab00a0bf7ae8714e56...
3,diagnostic_message,NaN
4,input_hash,b8229b1651d5bb63c584c53540e8f61c37152bbf471406...
5,policy_hash,NaN
6,postcheck_status,passed
7,round_results,[]
8,run_id,001_classical
9,run_status,success


Metriche complessive


,campo,valore
0,end_to_end_runtime_seconds,0.264035
1,recovery_attempt_count,0.000000
2,round_count,0.000000


## 2. Problema e dimensioni

Il solver opera sul problema originale, senza esclusioni euristiche.

In [7]:
display_vertical({
    "asset_count": len(artifacts.problem["asset_ids"]),
    "original_variable_counts": result["original_variable_counts"],
    "residual_variable_counts": result["residual_variable_counts"],
    "round_count": artifacts.reduction_round_count,
    "recovery_attempt_count": len(artifacts.recovery_attempts),
}, title="Dimensioni")

Dimensioni


,campo,valore
0,asset_count,30
1,original_variable_counts,"{\n ""binary_free"": 30,\n ""continuous_free"": ..."
2,residual_variable_counts,"{\n ""binary_free"": 30,\n ""continuous_free"": ..."
3,round_count,0
4,recovery_attempt_count,0


## 3. Soluzione e verifica

La certificazione dell'ottimalità dipende dall'esito del solver e del post-check; un run può terminare senza soluzione valida.

In [8]:
display_vertical({
    "final_retained_item_ids": result["final_retained_item_ids"],
    "cumulative_fixing_manifest": result["cumulative_fixing_manifest"],
    "classical_result": result["classical_result"],
    "postcheck_result": result["postcheck_result"],
    "original_problem_optimality_claim": result["original_problem_optimality_claim"],
}, title="Esito finale")

Esito finale


,campo,valore
0,final_retained_item_ids,"[\n ""asset-000"",\n ""asset-001"",\n ""asset-00..."
1,cumulative_fixing_manifest,"{\n ""entries"": [],\n ""excluded_item_ids"": []..."
2,classical_result,"{\n ""best_bound"": -0.10631980144575243,\n ""c..."
3,postcheck_result,"{\n ""failure_reasons"": [],\n ""fixed_variable..."
4,original_problem_optimality_claim,none
